### Face 1: Calculo de variable objetivo.

| Vamos a encontrar un tiempo optimo 

Importamos librerias

In [13]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import pyswarms as ps
tqdm.pandas()


In [ ]:
class PSO_SemaforoOptimizer:
    """
    Optimizador PSO ligero para tiempos de luz verde.
    """
    def __init__(self, n_particles=20, n_iterations=50):
        self.n_particles = n_particles
        self.n_iterations = n_iterations
        self.w, self.c1, self.c2 = 0.5, 1.5, 1.5
        
    def fitness_function(self, tiempo_verde, vehiculos, ocupacion, tiempo_medio, cluster):
        if tiempo_medio <= 0 or tiempo_verde <= 0: return 1e6
        
        capacidad = tiempo_verde / tiempo_medio
        ratio_demanda = vehiculos / max(capacidad, 0.1)
        
        deficit = max(0, ratio_demanda - 1) ** 2 * 100
        exceso = max(0, 1 - ratio_demanda) ** 2 * 30
        
        refs = {0: (55, 0.2), 1: (35, 0.15), 2: (20, 0.1)}
        t_ref, peso = refs.get(cluster, (33, 0.1))
        
        desviacion = abs(tiempo_verde - t_ref) * peso
        urgencia = (ocupacion - 50) * 2 if ocupacion > 50 and tiempo_verde < 25 else 0
        
        return deficit + exceso + (ocupacion/100 * deficit * 2) + desviacion + urgencia
    
    def get_bounds(self, cluster):
        return {0: (35, 90), 1: (20, 60), 2: (10, 40)}.get(cluster, (15, 60))
    
    def optimize(self, vehiculos, ocupacion, tiempo_medio, cluster):
        t_min, t_max = self.get_bounds(cluster)
        particles = np.random.uniform(t_min, t_max, self.n_particles)
        velocities = np.zeros(self.n_particles)
        
        pbest = particles.copy()
        pbest_fit = np.array([self.fitness_function(p, vehiculos, ocupacion, tiempo_medio, cluster) for p in particles])
        
        gbest = pbest[np.argmin(pbest_fit)]
        gbest_fit = np.min(pbest_fit)
        
        for _ in range(self.n_iterations):
            r1, r2 = np.random.random(2)
            velocities = (self.w * velocities + 
                          self.c1 * r1 * (pbest - particles) + 
                          self.c2 * r2 * (gbest - particles))
            particles = np.clip(particles + velocities, t_min, t_max)
            
            for i in range(self.n_particles):
                fit = self.fitness_function(particles[i], vehiculos, ocupacion, tiempo_medio, cluster)
                if fit < pbest_fit[i]:
                    pbest[i], pbest_fit[i] = particles[i], fit
                    if fit < gbest_fit:
                        gbest, gbest_fit = particles[i], fit
                        
        return round(gbest)

def procesar_y_guardar_pso(input_csv, output_csv='dataset_optimizado4.csv'):
    df = pd.read_csv(input_csv)
    
    df['Hora_Inicio'] = pd.to_datetime(
        df['Hora_Inicio'],
        format='%H:%M:%S',
        errors='coerce'
    ).dt.time
    df['franja'] = df['Hora_Inicio'].apply(lambda x: f"{x.hour:02d}:{(x.minute // 15) * 15:02d}")
    
    optimizer = PSO_SemaforoOptimizer()
    grupos = df.groupby(['Dia_Semana', 'Direccion', 'cluster', 'franja'])
    print(f"Optimizando {len(grupos)} escenarios de tráfico...")
    mapa_tiempos = {}
    
    for grupo_id, datos in grupos:
        t_opt = optimizer.optimize(
            vehiculos=datos['Total_Vehiculos'].mean(),
            ocupacion=datos['Ocupacion_Espacial_%'].mean(),
            tiempo_medio=datos['Tiempo_Medio_s'].mean(),
            cluster=grupo_id[2] 
        )
        mapa_tiempos[grupo_id] = t_opt

    df['Tiempo_Optimo'] = df.set_index(['Dia_Semana', 'Direccion', 'cluster', 'franja']).index.map(mapa_tiempos)
    
    df = df.drop(columns=['franja'])
    # df.to_csv(output_csv, index=False)
    print(f"✓ Proceso completado. Archivo guardado como: {output_csv}")

if __name__ == "__main__":
    procesar_y_guardar_pso('completo_clusters.csv')

Optimizando 1324 escenarios de tráfico...


(np.int64(1), np.int64(1), np.int64(0), '08:00')

np.float64(7.6)

(np.int64(1), np.int64(1), np.int64(0), '08:15')

np.float64(9.0)

...

(np.int64(1), np.int64(4), np.int64(1), '15:45')

np.float64(3.0)

(np.int64(1), np.int64(4), np.int64(2), '14:00')

np.float64(2.0)

KeyboardInterrupt: 